In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path
repo = Path(os.environ['SOUP_REPO'])
results = Path(os.environ['SOUP_RESULTS'])
print('Editable challenge repo:', repo)
print('Public data:', list((repo / 'data').glob('*.jsonl')))
print('Write your solution files under', repo / 'solution')




In [ ]:












def _install_soup_candidate():
    import ast
    import base64
    import hashlib
    import io
    import json
    import os
    from pathlib import Path, PurePosixPath
    import stat
    import tempfile
    import zipfile

    expected_solution_sha256 = 'f749d20c06ad60390bace705f443067f455e0b169b5441047c040ebb3f9bd885'
    expected_archive_sha256 = '237f15c4e1df17fce2de1d1cdad869c182ae7945d8851fc04bb45bd5aa4b8561'
    protected_names = ('swiglu.py', 'test_kernels.py')
    expected_settings = {'CHUNK_TOKENS': 256,
     'COMPACT_MLP_ACTIVATIONS': True,
     'FROZEN_STORAGE': True,
     'FUSED_LORA_EPILOGUE': False,
     'FUSED_NF4': True,
     'FUSED_ROTARY': False,
     'HEAD_NUMERICS': 'native',
     'HEAD_PROJECTION': 'torch',
     'SPARSE_BACKWARD': True}
    manifest = {'__init__.py': {'bytes': 1873,
                     'sha256': 'd6556e8e878ffc7cf63680459cde21aee05cef071953b9ffadce21ba15c63435'},
     'activation_storage.py': {'bytes': 7404,
                               'sha256': '95f66325e0361ced6e0c56ad06a027c66ac8d371969dc44a5840f746dfd21ca8'},
     'frozen_storage.py': {'bytes': 8584,
                           'sha256': 'c27dc833f9b7ceb15b3f7e96169c5a6572060abead71c093aac8625636aa0fe0'},
     'head_fused_t4.py': {'bytes': 26490,
                          'sha256': 'ac75f93ce5b46eeb564afd8c74409cbbeda36804c0399d639c159bf8779b14ce'},
     'head_matmul.py': {'bytes': 4134,
                        'sha256': 'c834bc001e3c6f9449ef4f5cd60d34aa961262e27c81e98e9a47186af5ec7e39'},
     'head_matmul_t4.py': {'bytes': 9051,
                           'sha256': 'b29ed64e0d1537f6682d2b75d5a979c8269030a306aa8f1dee263bb9e3be3134'},
     'integration.py': {'bytes': 12622,
                        'sha256': '3da67233c3917e5efc5e1811f096664a95e8500007533e5bb1f921c2bcbb7a2d'},
     'lora_epilogue.py': {'bytes': 9912,
                          'sha256': '124f3ff31c7f1328bfa2a0e7760fd6bd7b48d9daa654d1addef154760767964e'},
     'metric_transfer.py': {'bytes': 6052,
                            'sha256': '2b30a1c1ec61eca8841a9f2f970588f65ce2847b6b178ebd7d950995dba5a2d9'},
     'nf4.py': {'bytes': 10430,
                'sha256': '9e07a73d45f7b7119fc1f5d1d3422351e5d34e2fbd9612f7b90f56bc17ad697a'},
     'rmsnorm.py': {'bytes': 5869,
                    'sha256': 'aa55a21fd921d2f5478444b03a32694a329bb10bbbe4f3bf46e4e78f2804a93c'},
     'rotary.py': {'bytes': 13338,
                   'sha256': '8e8ef322d70eba6ac898a040a14d74b9dc8e276b3f93bd033414eeaf927b5429'},
     'settings.py': {'bytes': 1489,
                     'sha256': '17261889d0b049ddc41a2607e7106dc62e466069466344bcab885be84f9494de'},
     'streamed_head.py': {'bytes': 27782,
                          'sha256': '65ad6cada653b0d03d70375570b132c69f47a52ee8ff65ef0cb11042a63c9f19'},
     'swiglu.py': {'bytes': 3185,
                   'sha256': 'e665910faeca6091c958ed7a191604484a44216f51d6fa069f5f48220e7472e6'},
     'test_activation_storage.py': {'bytes': 14607,
                                    'sha256': '4a4b8b9819c26745fb10ef8228119981fec75012b140844c43552f0ac42291cf'},
     'test_frozen_storage.py': {'bytes': 11574,
                                'sha256': '87c98888cdb260d6e0e3e2c639ab5e9107a26da1708b8b005a2a4aa1d61ef07d'},
     'test_head_fused_t4.py': {'bytes': 24883,
                               'sha256': '335d684aa6fdc063d478265209e6ee05fcdf32fa7babe8d3e6c24644679d512d'},
     'test_head_matmul.py': {'bytes': 2743,
                             'sha256': '1ae409f55282fbd96d3c78845ab22cded87bf453149fbecbbfe7e3f62ec4c9bd'},
     'test_head_matmul_t4.py': {'bytes': 7100,
                                'sha256': '75931a45b9623e2ceb458904166218d31ffc68ccb08f129da5691cbdc961467d'},
     'test_integration.py': {'bytes': 30248,
                             'sha256': 'e8b5c5571d8baa987e38affa925defbf10de29ef025771fa20c74d41101e6aea'},
     'test_kernels.py': {'bytes': 1300,
                         'sha256': 'a45b842a4f22c1aefe404baeff1ac26511253452ecb0f633c985fbc9df8b252a'},
     'test_lora_epilogue.py': {'bytes': 15228,
                               'sha256': 'c7394c871558fe12ab684aca5e5872ecd18763467e2ace460ff9016c7efd6421'},
     'test_nf4.py': {'bytes': 10862,
                     'sha256': '2ee2f46a3011ab76c61675bef3b3e1ce40e0964d1737c92c36b71e8fd05c9a4f'},
     'test_rmsnorm.py': {'bytes': 3634,
                         'sha256': 'a0ef2638f9d7704c5da4154dcf9eac2b87069952fd3ff743da63ea84ddbdfded'},
     'test_rotary.py': {'bytes': 18180,
                        'sha256': 'a8659bbb8455c6f20868672a7a41e4c96c67b5a066cd443addd0067acf4faf37'},
     'test_streamed_head.py': {'bytes': 29925,
                               'sha256': '5ebbca158118591e39fca8b70ecf9f2485b287f07715573ab1c271acf21775d4'},
     'test_training_parity.py': {'bytes': 3303,
                                 'sha256': '412b9ebcd7d1d95c27b0e388a43f9b71820c3959ed5caa04345c09ba9fa945f1'}}
    payload_b64 = '...'


    archive_bytes = base64.b64decode(payload_b64, validate=True)
    if hashlib.sha256(archive_bytes).hexdigest() != expected_archive_sha256:
        raise RuntimeError("Embedded candidate archive checksum mismatch")
    payload = {}
    with zipfile.ZipFile(io.BytesIO(archive_bytes), "r") as archive:
        members = archive.infolist()
        names = [member.filename for member in members]
        if len(names) != len(set(names)) or set(names) != set(manifest):
            raise RuntimeError("Embedded archive has missing, duplicate, or unexpected entries")
        for member in members:
            name = member.filename
            path = PurePosixPath(name)
            if (path.is_absolute() or len(path.parts) != 1 or path.name != name
                    or "\\" in name or not name.endswith(".py")
                    or member.is_dir() or stat.S_ISLNK(member.external_attr >> 16)):
                raise RuntimeError("Unsafe embedded archive path: " + repr(name))
            expected = manifest[name]
            if member.file_size != expected["bytes"]:
                raise RuntimeError("Embedded source size mismatch: " + name)
            content = archive.read(member)
            if (len(content) != expected["bytes"]
                    or hashlib.sha256(content).hexdigest() != expected["sha256"]):
                raise RuntimeError("Embedded source checksum mismatch: " + name)
            compile(content, "solution/" + name, "exec")
            payload[name] = content


    settings = {}
    for node in ast.parse(payload["settings.py"]).body:
        if isinstance(node, ast.Assign):
            for lhs in node.targets:
                if isinstance(lhs, ast.Name):
                    settings[lhs.id] = ast.literal_eval(node.value)
    if settings != expected_settings:
        raise RuntimeError("Embedded candidate settings differ from the fixed candidate")
    entrypoints = [node for node in ast.parse(payload["__init__.py"]).body
                   if isinstance(node, ast.FunctionDef) and node.name == "apply"]
    if len(entrypoints) != 1:
        raise RuntimeError("Embedded candidate must define exactly one apply()")
    args = entrypoints[0].args
    if args.posonlyargs or args.args or args.kwonlyargs or args.vararg or args.kwarg:
        raise RuntimeError("Embedded candidate apply() must take no arguments")

    repo_value = os.environ.get("SOUP_REPO")
    if repo_value is None or not repo_value.strip():
        raise RuntimeError("Set SOUP_REPO to the existing official starter checkout first")
    repo = Path(repo_value).resolve(strict=True)
    if not repo.is_dir():
        raise RuntimeError("SOUP_REPO must name an existing directory")
    solution = repo / "solution"
    if solution.is_symlink() or not solution.is_dir():
        raise RuntimeError("SOUP_REPO/solution must be an existing real directory")


    unexpected = []
    for path in sorted(solution.rglob("*")):
        relative = path.relative_to(solution)
        name = str(relative)
        if path.is_symlink():
            raise RuntimeError("Refusing a symlink in solution/: " + name)
        if "__pycache__" in relative.parts:
            continue
        if not path.is_file() or name not in manifest:
            unexpected.append(name)
    if unexpected:
        raise RuntimeError("Unexpected solution/ entries; inspect them before retrying: "
                           + ", ".join(unexpected))
    for name in protected_names:
        path = solution / name
        if not path.is_file() or path.read_bytes() != payload[name]:
            raise RuntimeError("Protected starter source missing or different: solution/"
                               + name + "; no files were written")

    updated = []
    unchanged = []
    for name, content in sorted(payload.items()):
        destination = solution / name
        if name in protected_names:
            continue
        if destination.is_file() and destination.read_bytes() == content:
            unchanged.append(name)
            continue


        temporary = None
        try:
            with tempfile.NamedTemporaryFile(mode="wb", dir=solution,
                                             prefix="._soup_install_", suffix=".tmp",
                                             delete=False) as handle:
                temporary = Path(handle.name)
                handle.write(content)
                handle.flush()
                os.fchmod(handle.fileno(), 0o644)
            os.replace(temporary, destination)
            temporary = None
            updated.append(name)
        finally:
            if temporary is not None:
                try:
                    temporary.unlink()
                except FileNotFoundError:
                    pass



    digest = hashlib.sha256()
    for path in sorted(Path(solution).rglob("*")):
        if path.is_file() and "__pycache__" not in path.parts:
            digest.update(str(path.relative_to(solution)).encode())
            digest.update(path.read_bytes())
    observed_solution_sha256 = digest.hexdigest()
    if observed_solution_sha256 != expected_solution_sha256:
        raise RuntimeError("Installed whole-solution hash mismatch: "
                           + observed_solution_sha256)
    print(json.dumps({
        "status": "installed_and_verified",
        "solution_sha256": observed_solution_sha256,
        "source_files": len(manifest),
        "updated_files": len(updated),
        "already_matching_files": len(unchanged),
        "protected_files_verified": list(protected_names),
        "settings": settings,
        "apply_called": False,
    }, indent=2, sort_keys=True))


_install_soup_candidate()


In [ ]:



from pathlib import Path
import hashlib

nf4_path = repo / "solution" / "nf4.py"
init_path = repo / "solution" / "__init__.py"
nf4_src = nf4_path.read_text(encoding="utf-8")
init_src = init_path.read_text(encoding="utf-8")
assert hashlib.sha256(nf4_src.encode()).hexdigest() == "9e07a73d45f7b7119fc1f5d1d3422351e5d34e2fbd9612f7b90f56bc17ad697a"
assert hashlib.sha256(init_src.encode()).hexdigest() == "d6556e8e878ffc7cf63680459cde21aee05cef071953b9ffadce21ba15c63435"

nf4_src += r'''

_NF4_WEIGHT_CACHE = {}
_NF4_WEIGHT_CACHE_BYTES = 0
_NF4_WEIGHT_CACHE_LIMIT = 1 * 1024**3


def _cache_tensor_signature(tensor):
    if tensor is None:
        return None
    storage = tensor.untyped_storage()
    return (
        storage.data_ptr(), storage.nbytes(), tensor.storage_offset(),
        tuple(tensor.shape), tuple(tensor.stride()), tensor.dtype,
        tensor.device, tensor._version,
    )


def _cache_components_key(components):
    return (
        _cache_tensor_signature(components["packed"]),
        tuple(components["shape"]), components["dtype"], components["blocksize"],
        _cache_tensor_signature(components["scales"]),
        _cache_tensor_signature(components.get("nested_scales")),
        _cache_tensor_signature(components.get("nested_code")),
        _cache_tensor_signature(components.get("offset")),
        components.get("nested_blocksize", 256),
    )


def _cache_stream_supported(device):
    if torch.cuda.is_current_stream_capturing():
        return False
    current = torch.cuda.current_stream(device)
    default = torch.cuda.default_stream(device)
    return current.cuda_stream == default.cuda_stream


def _cache_decoded_weight(components):
    global _NF4_WEIGHT_CACHE_BYTES
    if components.get("out") is not None or components["dtype"] != torch.float16:
        return None
    packed = components["packed"]
    if (not packed.is_cuda or packed.requires_grad
            or not _cache_stream_supported(packed.device)):
        return None

    key = _cache_components_key(components)
    entry = _NF4_WEIGHT_CACHE.get(key)
    if entry is not None:
        return entry[0]

    decoded = _execute(**components)
    size = decoded.numel() * decoded.element_size()
    if _NF4_WEIGHT_CACHE_BYTES + size > _NF4_WEIGHT_CACHE_LIMIT:
        return decoded

    # Hold the immutable packed/metadata tensors as ownership guards so a later
    # allocator reuse cannot alias an entry with the same raw addresses.
    refs = (
        components["packed"], components["scales"],
        components.get("nested_scales"), components.get("nested_code"),
        components.get("offset"),
    )
    _NF4_WEIGHT_CACHE[key] = (decoded, refs)
    _NF4_WEIGHT_CACHE_BYTES += size
    return decoded


def patch_nf4_weight_cache():
    """Reuse exact decoded frozen NF4 weights at both reviewed bnb boundaries."""
    import bitsandbytes.functional as functional
    from bitsandbytes.backends.cuda import ops

    if getattr(functional.dequantize_4bit, "_soup_full_nf4_cache", False):
        return True
    if not getattr(functional.dequantize_4bit, "_soup_candidate_nf4", False):
        return False

    original_dequantize = functional.dequantize_4bit
    original_fallback = ops._dequant_linear_fallback

    @wraps(original_dequantize)
    def cached_dequantize_4bit(A, quant_state=None, absmax=None, out=None,
                               blocksize=None, quant_type="fp4"):
        components = _state_components(A, quant_state, out, cuda_only=True)
        if components is None:
            return original_dequantize(
                A, quant_state, absmax, out, blocksize, quant_type
            )
        decoded = _cache_decoded_weight(components)
        if decoded is None:
            return original_dequantize(
                A, quant_state, absmax, out, blocksize, quant_type
            )
        return decoded.t() if A.shape[0] == 1 else decoded

    @wraps(original_fallback)
    def cached_dequant_linear_fallback(
        A, B, shapeB, absmax, blocksize, quant_type,
        bias=None, absmax_8bit=None, absmax_code=None, absmax_offset=None
    ):
        nested = absmax_8bit is not None
        components = dict(
            packed=B, shape=shapeB, dtype=A.dtype,
            scales=absmax_8bit if nested else absmax,
            blocksize=blocksize,
            nested_scales=absmax if nested else None,
            nested_code=absmax_code,
            offset=absmax_offset,
            nested_blocksize=256,
        )
        if (quant_type != "nf4" or A.device != B.device
                or not _components_supported(**components, cuda_only=True)):
            return original_fallback(
                A, B, shapeB, absmax, blocksize, quant_type, bias,
                absmax_8bit, absmax_code, absmax_offset
            )
        decoded = _cache_decoded_weight(components)
        if decoded is None:
            return original_fallback(
                A, B, shapeB, absmax, blocksize, quant_type, bias,
                absmax_8bit, absmax_code, absmax_offset
            )
        return torch.nn.functional.linear(A, decoded, bias)

    cached_dequantize_4bit._soup_candidate_nf4 = True
    cached_dequantize_4bit._soup_full_nf4_cache = True
    cached_dequant_linear_fallback._soup_candidate_nf4 = True
    cached_dequant_linear_fallback._soup_full_nf4_cache = True
    functional.dequantize_4bit = cached_dequantize_4bit
    ops._dequant_linear_fallback = cached_dequant_linear_fallback
    return True
'''

anchor = '''        from .nf4 import patch_bitsandbytes_nf4
        patch_bitsandbytes_nf4()
'''
replacement = '''        from .nf4 import patch_bitsandbytes_nf4, patch_nf4_weight_cache
        patch_bitsandbytes_nf4()
        patch_nf4_weight_cache()
'''
assert init_src.count(anchor) == 1
init_src = init_src.replace(anchor, replacement, 1)

compile(nf4_src, str(nf4_path), "exec")
compile(init_src, str(init_path), "exec")
nf4_path.write_text(nf4_src, encoding="utf-8")
init_path.write_text(init_src, encoding="utf-8")
print("full_nf4_cache_patch=installed")
print("nf4_sha256=", hashlib.sha256(nf4_src.encode()).hexdigest())
print("init_sha256=", hashlib.sha256(init_src.encode()).hexdigest())


In [ ]:


from pathlib import Path
import hashlib

init_path = repo / "solution" / "__init__.py"
init_src = init_path.read_text(encoding="utf-8")
assert init_path.is_file()

padding_src = r'''"""Within-batch Qwen2 MLP padding compaction for the fixed Soup workload."""

import contextvars
import inspect
import torch

_ACTIVE_ROWS = contextvars.ContextVar("_soup_mlp_active_rows", default=None)


def patch_mlp_padding_compaction():
    from transformers.models.qwen2 import modeling_qwen2

    mlp_cls = modeling_qwen2.Qwen2MLP
    model_cls = modeling_qwen2.Qwen2Model
    if getattr(mlp_cls.forward, "_soup_padding_compaction", False):
        return True

    original_mlp = mlp_cls.forward
    original_model = model_cls.forward
    model_signature = inspect.signature(original_model)

    def model_forward(self, *args, **kwargs):
        try:
            bound = model_signature.bind_partial(self, *args, **kwargs)
            mask = bound.arguments.get("attention_mask")
        except TypeError:
            mask = kwargs.get("attention_mask")

        plan = None
        if (isinstance(mask, torch.Tensor) and mask.is_cuda and mask.ndim == 2
                and mask.numel() > 0):
            flat_mask = mask.reshape(-1) != 0
            active = torch.nonzero(flat_mask, as_tuple=False).flatten()
            # The tensor shape of nonzero is known after the op; no value-dependent
            # CPU branching is used to choose training examples or reorder batches.
            if active.numel() < flat_mask.numel():
                plan = (active, tuple(mask.shape))

        token = _ACTIVE_ROWS.set(plan)
        try:
            return original_model(self, *args, **kwargs)
        finally:
            _ACTIVE_ROWS.reset(token)

    def mlp_forward(self, x):
        plan = _ACTIVE_ROWS.get()
        if (plan is None or not isinstance(x, torch.Tensor) or not x.is_cuda
                or x.ndim != 3):
            return original_mlp(self, x)

        active, mask_shape = plan
        if tuple(x.shape[:2]) != mask_shape or active.device != x.device:
            return original_mlp(self, x)

        flat = x.reshape(-1, x.shape[-1])
        compact = flat.index_select(0, active)
        compact_out = original_mlp(self, compact)
        out = torch.zeros_like(flat)
        out.index_copy_(0, active, compact_out.reshape(active.numel(), -1))
        return out.view(*x.shape[:-1], compact_out.shape[-1])

    model_forward._soup_padding_compaction = True
    mlp_forward._soup_padding_compaction = True
    model_cls.forward = model_forward
    mlp_cls.forward = mlp_forward
    return True
'''

padding_path = repo / "solution" / "mlp_padding.py"
compile(padding_src, str(padding_path), "exec")
padding_path.write_text(padding_src, encoding="utf-8")

anchor = '''    if COMPACT_MLP_ACTIVATIONS:
        from .activation_storage import patch_compact_mlp_activations
        patch_compact_mlp_activations()
'''
replacement = anchor + '''    from .mlp_padding import patch_mlp_padding_compaction
    patch_mlp_padding_compaction()
'''
assert init_src.count(anchor) == 1
init_src = init_src.replace(anchor, replacement, 1)
compile(init_src, str(init_path), "exec")
init_path.write_text(init_src, encoding="utf-8")

print("mlp_padding_patch=installed")
print("mlp_padding_sha256=", hashlib.sha256(padding_src.encode()).hexdigest())
print("init_sha256=", hashlib.sha256(init_src.encode()).hexdigest())


In [ ]:

from pathlib import Path
import os, hashlib
_solution_root = Path(os.environ["SOUP_REPO"]) / "solution"
assert (_solution_root / "mlp_padding.py").exists(), "MLP compaction patch missing"
_head_path = _solution_root / "streamed_head.py"
assert hashlib.sha256(_head_path.read_bytes()).hexdigest() == "65ad6cada653b0d03d70375570b132c69f47a52ee8ff65ef0cb11042a63c9f19", "Unexpected head source; refuse composite"
assert "patch_nf4_weight_cache()" in (_solution_root / "__init__.py").read_text()
assert "patch_mlp_padding_compaction()" in (_solution_root / "__init__.py").read_text()
print("base=storage-f749d20, overlays=full-NF4-cache+MLP-pad, adding exact native-logits reuse")








from pathlib import Path
import hashlib
head_src = repo / "solution" / "streamed_head.py"
settings_src = repo / "solution" / "settings.py"
original = head_src.read_text(encoding="utf-8")
assert settings_src.read_text(encoding="utf-8").count("CHUNK_TOKENS = 256") == 1
start = original.index("class _NativeStreamedLinearCrossEntropy(")
end = original.index("\nclass _StreamedLinearCrossEntropy(", start)
before, body, after = original[:start], original[start:end], original[end:]
edits = [
    ("        predictions = torch.empty(token_count, device=hidden.device, dtype=torch.long)\n",
     "        predictions = torch.empty(token_count, device=hidden.device, dtype=torch.long)\n"
     "        # Retain the original FP16 projected scores for exact backward replay.\n"
     "        saved_logits = torch.empty((token_count, weight_half.shape[0]),\n"
     "                                   device=hidden.device, dtype=torch.float16)\n"),
    ("                scores = torch.mm(flat_hidden[start:end], weight_half.t())\n",
     "                scores = saved_logits[start:end]\n"
     "                torch.mm(flat_hidden[start:end], weight_half.t(), out=scores)\n"),
    ("        ctx.save_for_backward(flat_hidden, weight_half, targets, denominator, unsafe_ignored_rows)\n",
     "        ctx.save_for_backward(flat_hidden, weight_half, targets, denominator,\n"
     "                              unsafe_ignored_rows, saved_logits)\n"),
    ("        flat_hidden, weight_half, targets, denominator, unsafe_ignored_rows = ctx.saved_tensors\n",
     "        (flat_hidden, weight_half, targets, denominator,\n"
     "         unsafe_ignored_rows, saved_logits) = ctx.saved_tensors\n"),
    ("                scores = torch.mm(work_hidden[start:end], weight_half.t())\n",
     "                if selected_rows is None:\n"
     "                    scores = saved_logits[start:end]\n"
     "                else:\n"
     "                    scores = saved_logits.index_select(0, selected_rows[start:end])\n"),
]
for old, new in edits:
    assert body.count(old) == 1, "Unexpected native head source"
    body = body.replace(old, new, 1)
modified = before + body + after
compile(modified, str(head_src), "exec")
head_src.write_text(modified, encoding="utf-8")
assert head_src.read_text(encoding="utf-8") == modified
print("C0 forward-logits reuse installed; SHA256:", hashlib.sha256(modified.encode()).hexdigest())


In [ ]:


from pathlib import Path
import ast
import hashlib
q_source = "\"\"\"Qwen pad-elision experiment for the PEFT q_proj only.\n\nValid-row projection uses original bitsandbytes matmul and LoRA math; dropout\nis sampled on full B*T geometry to conserve the generator state and masks.\n\"\"\"\nimport torch\n\ndef patch_query_projection_compaction():\n    from transformers.models.qwen2.modeling_qwen2 import Qwen2Attention\n    from peft.tuners.lora.bnb import Linear4bit\n    from .mlp_padding import _ACTIVE_ROWS\n    if getattr(Qwen2Attention.forward, \"_soup_query_padding\", False):\n        return True\n\n    original_attention_forward = Qwen2Attention.forward\n    original_lora_forward = Linear4bit.forward\n\n    def compact_query_forward(self, x, *args, **kwargs):\n        plan = getattr(self, \"_soup_query_padding_plan\", None)\n        if plan is None:\n            return original_lora_forward(self, x, *args, **kwargs)\n        active, mask_shape = plan\n        if (args or kwargs or not self.training or self.disable_adapters or self.merged\n                or not isinstance(x, torch.Tensor) or x.ndim != 3 or not x.is_cuda\n                or tuple(x.shape[:2]) != mask_shape or active.device != x.device\n                or active.ndim != 1 or active.dtype != torch.long\n                or active.numel() == 0 or active.numel() == x.shape[0]*x.shape[1]\n                or len(self.active_adapters) != 1):\n            return original_lora_forward(self, x, *args, **kwargs)\n        adapter = self.active_adapters[0]\n        if adapter not in self.lora_A or adapter in self.lora_variant:\n            return original_lora_forward(self, x, *args, **kwargs)\n        self._check_forward_args(x, *args, **kwargs)\n        dim=x.shape[-1]\n        compact_x=x.reshape(-1,dim).index_select(0,active)\n        base=self.base_layer(compact_x).clone()\n        lora_A=self.lora_A[adapter]\n        lora_B=self.lora_B[adapter]\n        dropout=self.lora_dropout[adapter]\n        scale=self.scaling[adapter]\n        requires_conversion=not torch.is_autocast_enabled()\n        dropout_x=self._cast_input_dtype(x,lora_A.weight.dtype) if requires_conversion else x\n        # Do not generate dropout on compact x: the full-mask RNG sequence is required.\n        dropout_full=dropout(dropout_x)\n        dropout_live=dropout_full.reshape(-1,dim).index_select(0,active)\n        lora_delta=lora_B(lora_A(dropout_live))*scale\n        if requires_conversion:\n            lora_delta=lora_delta.to(base.dtype)\n        live=base+lora_delta\n        flat=torch.zeros((x.shape[0]*x.shape[1],live.shape[-1]),dtype=live.dtype,device=live.device)\n        flat=flat.index_copy(0,active,live)\n        return flat.view(x.shape[0],x.shape[1],live.shape[-1])\n\n    def attention_forward(self,*args,**kwargs):\n        plan=_ACTIVE_ROWS.get()\n        q=self.q_proj\n        if (not self.training or plan is None or not isinstance(q,Linear4bit)\n                or getattr(q,\"_soup_query_padding_plan\",None) is not None):\n            return original_attention_forward(self,*args,**kwargs)\n        q._soup_query_padding_plan=plan\n        try:\n            return original_attention_forward(self,*args,**kwargs)\n        finally:\n            del q._soup_query_padding_plan\n\n    compact_query_forward._soup_query_padding=True\n    attention_forward._soup_query_padding=True\n    Linear4bit.forward=compact_query_forward\n    Qwen2Attention.forward=attention_forward\n    return True\n"
q_path = repo / "solution" / "q_padding.py"
assert not q_path.exists(), "Refusing to overwrite existing q_padding.py"
ast.parse(q_source, filename=str(q_path))
q_path.write_text(q_source)
init_path = repo / "solution" / "__init__.py"
init_text = init_path.read_text()
anchor = '    from .mlp_padding import patch_mlp_padding_compaction\n    patch_mlp_padding_compaction()\n'
assert init_text.count(anchor) == 1, "Unexpected base padding source"
init_text = init_text.replace(anchor, anchor+'    from .q_padding import patch_query_projection_compaction\n    patch_query_projection_compaction()\n', 1)
ast.parse(init_text, filename=str(init_path))
init_path.write_text(init_text)
print('qproj_padding_installed', hashlib.sha256(q_source.encode()).hexdigest())


In [ ]:


from pathlib import Path
import os,base64,zlib,hashlib,json
repo=Path(os.environ['SOUP_REPO'])
module=zlib.decompress(base64.b64decode('eNqdV1mP2zYQftevYJ0XeWEz2U2Qh0VdNMgBFMjVxGgeikCgpZHNLk1qScpHfn1nSFGWj90AERZriZwZznxzcjQafbJyKbVQbCv9SurpQvhyxYT3oL00mpnWN62fNtb8B2VYKc26EfEV/xpRVVCx+xasBMez7HXcdrip9uzvLeibV0kaNwUJmjDBamt+ALIbJ2lnKx2wj+9eMCX2YDl7pVTWKzFh9083TFSi8WBZZU2DWk2YN3cowdgK7IStTQXqqTLOMQfeS710TOiKmcbLtfwBNjMNWEHiHLOwFlIzjZ8b4Owz2oAMvTLIaYG5khSwaJxwDAWYKNCvIDMJtAEs0rHWIfF2BRo2qCcSIlbaw86jksytRAOBSru2aYz1UCFcc6QiNDcwjfY0SgRhZuFRRxSISK2DMNhJR4ax3EEj0BZAgDdCyQpfq3H24f3nadOZQgdboxRhOV+huMqAYx8/zVm5EnoJQeBX0zas4yhqC2gzRkG54tloNMrCuUVRt761UBRMrklpxEAbH3HMIk3d6tIbo1wi2VrRuKz78MaWqyzLKqhZQTtFDIIIW354Hd9mDB9ZsyXgAd4O9iZs1LEVSd9DGI4m7J1QDjoB9FhApXVcDotBUb5WTeJPuhavXs//+udt8eXTt6+BsvftbOBdXhu7FbbKAsmfwcA8UY7DItnXUeW7CbsSdunw5+puS28D3Z4w8vlR4B+HkdQVNID/NKJdWgrpEBqOB04LQg2E3bxha+HuiLGx4MBuMGYW++BgjIhBJJB6uIyEIVmStrwXFkJvdgQJR1/k454CnZOnAP1oNFBgk3n0Gw3Ft546PRgwyCC180KXQOiEoOBztMnYMTETyY5LV5RtJS7J2HFdyTX7bcaeEz1+tmtQ+ZjNZuzZsQQ0WC5b07p8fFlS5feYikErzfKoS62M8Ncvk2rh8/nNeOC3QVwlz19ydE8fs3oSvFPE7J8FiI/QPMEmMZ0BFDd4BRtZAsGwS+8XTPRtoxBnHk799/bm+5g4BoocBCZUry/JSTQd1L/P2MsXQ95u/Q/SJp717Du76j+uv/8CfD1DrYRnJBmjmsTl0+tJL3qKsnvKrhQgMfFwyp9d4UBhSuXPJp22Z+QFNhFk6ZXplsePuWfAeu6jM/gG1D3ON4TfcKPHDTePcT0B7wl7AzUehhRd55rCfSuxAVChqIVSC1HehSQftJiugTt+Kov6AVmngTocNk3Mmyo1V1aiuFsWSy4le9+wFfYkYfkvZ0UEPUIXmmqeB6clGCYXwCFnT87QPTwhn2dDvrAyYTFDZilVjrTowqQ0zb44RMnR8ePTfkJsGwnb/KrPLdTtIZVjKJ83EYrTrp2cEDzc45Bnblv4GcOgeaXXbKB/EBFbcQhppZATGz5gqYzBho2foszS2NORhF7PRE2z130rsLj+ELEnd/2Gumkt7nCmAIejzoKUc5xGiL7zOpw0ilJJ7i0NNZa72qcO/PXdfB5Xv2FbxSHtwIbU2iFWa7COh6bl+D1NlPGDLA6fSdTxtHmQs5De4ey22KOGXOtE/j7E8gvcja4yW1QCobusUT9+zfoBJTBcmE1qdJqx+wgujifUKsdpuunlyJiAtHk7LDpBKscChmW8GEKQOJIELLAn9VXQNPGlRfPX8NZaY/PRa6NrJctwYsBnOG10irKo6Fm8p4NOB6MHNfzJfBQY8m0E9bEZCQOrVUfV+UGmnicONQfn9ByjsHPkhg7pQWUPNKmma80/mKpVcFKCqbauwwZNDoGFx+8+gYYPnkB1KI8kY3LfcYTeXixqoebOuoO6K9NFwjhJ1nKHyeppBuzSEyt7V627Ar4FuVx5GiPpjsUfEIZ3LmZwbKSRrlxJjzGCk797mqRaujj5w8UpNAl3WVoynY4Phh9yLdyiKIyF3uePVHUEAls/djjUoVhaUQX4G0KepHK6Aq0Bq5I7a5XD54E7x1msx5AbFIKLGTiLMRyo8CJVKRrq+kifxN3xUMhjpQF5o5Bhke5W/ge+BwO7'))
assert hashlib.sha256(module).hexdigest()=='8ff39c5583cb0f5e94c7ce09d81f3dae9dc74322a4b4c0d51b7c14111bd84b70'
modpath=repo/'solution'/'attention_o_proj_compaction_t4.py'
modpath.write_bytes(module)
init=repo/'solution'/'__init__.py'
text=init.read_text()
marker='# o_proj_padding_compaction_v1'
if marker not in text:
    text += "\n\n"+marker+"\n_before_o_proj_apply=apply\ndef apply():\n    _before_o_proj_apply()\n    from .attention_o_proj_compaction_t4 import install_patches\n    install_patches()\n"
    compile(text,str(init),'exec')
    init.write_text(text)


import torch
import bitsandbytes as bnb
torch.manual_seed(76033)
assert torch.cuda.get_device_name(0)=='Tesla T4'
h,o=64,48
proj=bnb.nn.Linear4bit(h,o,bias=False,compute_dtype=torch.float16,
                       quant_type='nf4',compress_statistics=True).to('cuda')
b,t=4,200
mask=(torch.arange(t,device='cuda')[None,:] <
      torch.tensor([130,100,200,75],device='cuda')[:,None])
active=torch.nonzero(mask.reshape(-1),as_tuple=False).flatten()
x=torch.randn((b,t,h),device='cuda',dtype=torch.float16,requires_grad=True)
xx=x.detach().clone().requires_grad_(True)
native=proj(x)
compact=proj(xx.reshape(-1,h).index_select(0,active))
out=torch.zeros((b*t,o),device='cuda',dtype=compact.dtype)
out.index_copy_(0,active,compact)
out=out.view(b,t,o)
torch.testing.assert_close(native[mask],out[mask],rtol=5e-3,atol=3e-3)
upstream=torch.randn_like(native)*mask.unsqueeze(-1)
gref=torch.autograd.grad(native,x,upstream)[0]
gnew=torch.autograd.grad(out,xx,upstream)[0]
torch.testing.assert_close(gref,gnew,rtol=6e-3,atol=4e-3)
path=Path(os.environ['SOUP_RESULTS'])
path.mkdir(parents=True,exist_ok=True)
path.joinpath('o_proj_padding_cuda_oracle.json').write_text(json.dumps({
  'status':'pass','source_sha256':'8ff39c5583cb0f5e94c7ce09d81f3dae9dc74322a4b4c0d51b7c14111bd84b70',
  'active_positions':int(active.numel()),'total_positions':b*t,
  'max_forward_error':float((native[mask]-out[mask]).abs().max().item()),
  'max_grad_error':float((gref-gnew).abs().max().item()),
  'real_NF4':True},indent=2))
print('PASS original o_proj compaction CUDA preflight')


In [ ]:

run = Path('/kaggle/working/training-run')
subprocess.run([sys.executable, str(repo / 'harness/run_task.py'), '--out', str(run), '--timeout', '900', '--only', 'solution'], cwd=repo, check=True)


In [ ]:

for path in run.rglob('*'):
    if path.is_file() and path.suffix in ('.json', '.jsonl', '.log', '.yaml') and path.stat().st_size < 4 * 1024**2:
        target = results / path.relative_to(run)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
print((run / 'result.json').read_text())



In [ ]:



import json, runpy, os
from pathlib import Path
_checker = runpy.run_path(str(repo / "harness" / "check_rules.py"),
                          run_name="_source_attribution_only")
_own, _copies, _lookalikes, _banned, _broken = _checker["source_kernels"](repo / "solution")
_result=json.loads((run / "result.json").read_text())
_sol_runs=_result["runs"]["solution"]
assert len(_sol_runs)==1 and _sol_runs[0].get("exit_code")==0
_met=_sol_runs[0]["metrics"]
_files=_met.get("triton_files", {})
_launches=_met.get("triton_launches", {})
_kernel_ms=_met.get("own_kernel_ms", {})
_allow=[
    n for n in _own if n in _launches and _launches[n]>0 and
    str(_files.get(n,"")).replace("\\","/").startswith("solution/")
]
_credited={n:float(_kernel_ms[n]) for n in _allow if n in _kernel_ms}
_timed=float(_met["timed_loop_s"])
assert _timed>0
_share=sum(_credited.values())/(_timed*1000.)
_audit={
    "kind":"pinned_official_source_kernel_timing_steps3plus",
    "useful_source_files":{n:_own[n] for n in _allow},
    "credited_kernel_ms":_credited,
    "timed_loop_s":_timed,
    "credited_original_share_percent":100*_share,
    "passed_minimum_5pct":_share>=0.05,
    "example_kernels_excluded":_copies,
    "near_example_kernels":_lookalikes,
    "banned_source_imports":_banned,
    "source_parse_errors":_broken,
    "source":"harness/check_rules.py; harness/_train.py",
    "note":"Single public solution run; not hidden jury or paired baseline admission.",
}
(results/"original_kernel_share.json").write_text(json.dumps(_audit,indent=2)+"\n")
print("ORIGINAL_TRITON_SHARE_PERCENT",round(100*_share,5))
print("ORIGINAL_TRITON_KERNELS",json.dumps(_credited))
print("TRITON_SOURCE_IMPORT_ISSUES", json.dumps(_banned+_broken))


In [ ]:







import base64, hashlib, json, os, re, subprocess, sys
from pathlib import Path
_bundle_base64 = "..."
_bundle_expected_sha256 = "bc188526e5a545727bd74588e092b7532a6dc5ea42283162c9091f8b9aa5f2da"
_bundle = Path("/kaggle/working/soup_test_only_pytest_bundle.zip")
_zipbytes = base64.b64decode(_bundle_base64, validate=True)
assert hashlib.sha256(_zipbytes).hexdigest() == _bundle_expected_sha256
_bundle.write_bytes(_zipbytes)
_testenv = dict(
    os.environ,
    PYTHONPATH=str(_bundle) + os.pathsep + os.environ.get("PYTHONPATH",""),
    PYTEST_DISABLE_PLUGIN_AUTOLOAD="1",
    PYTHONDONTWRITEBYTECODE="1",
    SOUP_APPLY_SOLUTION="1",
    PY_COLORS="0",
)
_cmd = [sys.executable,"-m","pytest","tests","-q","-p","no:cacheprovider","--color=no","--tb=short"]
_p = subprocess.run(_cmd, cwd=repo, env=_testenv, text=True,
                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                    check=False,timeout=1600)
_log = _p.stdout
(results / "soup79_bundle_test.log").write_text(_log[-300000:],encoding="utf-8")
_summary = {
    "kind":"postrun_unmodified_official_test_sources_with_bundled_pytest",
    "training_benchmark_unchanged": True,
    "bundle_sha256":_bundle_expected_sha256,
    "command":"PYTHONPATH=bundled_pytest SOUP_APPLY_SOLUTION=1 python -m pytest tests -q -p no:cacheprovider --color=no --tb=short",
    "returncode":_p.returncode,
    "exact_79_passed":bool(re.search(r"(?<!\d)79 passed\b",_log)) and _p.returncode==0,
    "tail":_log[-12000:],
}
(results / "soup79_bundle_test.json").write_text(json.dumps(_summary,indent=2)+"\n",encoding="utf-8")
print("BUNDLED_79_REGRESSIONS_RETURN_CODE:",_p.returncode)
print(_log[-3000:])
if not _summary["exact_79_passed"]:
    raise RuntimeError("OFFICIAL_79_TEST_SUITE_NOT_PASSED: " + "\n".join(_log.splitlines()[-36:])[-3600:])
print("79 official Soup regression tests passed with exact source and enabled solution.apply().")


In [ ]:


if _banned or _broken:
    raise RuntimeError("OWN_SOURCE_DISQUALIFICATION: "+repr(_banned+_broken))
if _share < 0.05:
    raise RuntimeError("OWN_TRITON_SHARE_LT_5_PERCENT: "+
                       str(round(100*_share,5))+
                       "% measured steps3+; required >=5%")
print("PRELIMINARY_POSTRUN_TESTS_PASS: 79 Soup regressions and "+
      str(round(100*_share,4))+"% original Triton share")
